In [11]:
import json
import pandas as pd
from pathlib import Path

LOG_PATH = Path("E3.log")
OUTPUT_PATH = Path("scene_evaluation_responses_E3.csv")

rows = []
with open(LOG_PATH, "r") as f:
    for line in f:
        if "scene_evaluation_completed" in line:
            try:
                # 로그 한 줄에서 JSON 파트 추출
                log_data = json.loads(line.split(" - INFO - ")[1])
                details = log_data["details"]

                scene_id = details["sceneId"]
                system_type = details["systemType"]
                system_name = details["systemName"]
                responses = details["responses"]

                # responses = { "1": "yes", "2": "no", ... }
                for q_id, ans in responses.items():
                    rows.append(
                        {
                            "sceneId": scene_id,
                            "systemType": system_type,
                            "systemName": system_name,
                            "questionId": int(q_id),
                            "answer": ans,
                        }
                    )
            except Exception as e:
                print("Parse error:", e)

# DataFrame으로 변환
df = pd.DataFrame(rows)

# CSV 저장
df.to_csv(OUTPUT_PATH, index=False)

print(f"✅ Saved {len(df)} responses to {OUTPUT_PATH}")
print(df.head(20))

✅ Saved 456 responses to scene_evaluation_responses_E3.csv
    sceneId systemType systemName  questionId answer
0         1    system1   Baseline           1    yes
1         1    system1   Baseline           2     no
2         1    system1   Baseline           3     no
3         1    system1   Baseline           4    yes
4         1    system1   Baseline           5    yes
5         1    system1   Baseline           6     no
6         1    system1   Baseline           7    yes
7         1    system1   Baseline           8     no
8         1    system1   Baseline           9    yes
9         1    system1   Baseline          10    yes
10        1    system1   Baseline          11     no
11        1    system1   Baseline          12     no
12        1    system1   Baseline          13    yes
13        1    system1   Baseline          14     no
14        1    system1   Baseline          15     no
15        1    system1   Baseline          16    yes
16        1    system1   Baseline       

In [ ]:

import pandas as pd

PATH = "scene_evaluation_responses_E3.csv"

# 데이터 불러오기
df = pd.read_csv(PATH)


# === 1. 질문 필터링 ===
def filter_questions(row):
    scene_id = row["sceneId"]
    q_id = row["questionId"]
    if scene_id <= 6 and q_id in [5, 6, 19, 20, 21]:
        return False
    if scene_id >= 7 and q_id in [19, 20, 21]:
        return False
    return True


df_filtered = df[df.apply(filter_questions, axis=1)].copy()

# === 2. sceneId, systemType별 "yes" 비율 계산 ===
yes_ratio_df = (
    df_filtered.groupby(["sceneId", "systemType"])["answer"]
    .apply(lambda x: (x == "yes").mean())
    .reset_index()
    .rename(columns={"answer": "yes_ratio"})
)


# === 3. participant_id 매핑 ===
def assign_participant_id(row):
    scene_id = row["sceneId"]
    system = row["systemType"]
    if system == "system1":
        return scene_id
    else:  # system2
        if scene_id <= 6:
            return scene_id + 6  # 1~6 → 7~12
        else:
            return scene_id - 6  # 7~12 → 1~6


yes_ratio_df["participant_id"] = yes_ratio_df.apply(assign_participant_id, axis=1)
yes_ratio_df.sort_values(["participant_id", "systemType"])


,sceneId,systemType,yes_ratio,participant_id
0,1,system1,0.473684,1
13,7,system2,0.789474,1
2,2,system1,0.578947,2
15,8,system2,0.842105,2
4,3,system1,0.684211,3
17,9,system2,0.578947,3
6,4,system1,0.947368,4
19,10,system2,0.789474,4
8,5,system1,0.684211,5
21,11,system2,0.631579,5


In [13]:
from scipy import stats

# === 4. participant별 system1 vs system2 매칭 ===
pivot_df = yes_ratio_df.pivot(
    index="participant_id", columns="systemType", values="yes_ratio"
).reset_index()

# === 5. Paired t-test ===
t_stat, p_val = stats.ttest_rel(pivot_df["system1"], pivot_df["system2"])

print(pivot_df)
print(f"Paired t-test: t = {t_stat:.3f}, p = {p_val:.4f}")

systemType  participant_id   system1   system2
0                        1  0.473684  0.789474
1                        2  0.578947  0.842105
2                        3  0.684211  0.578947
3                        4  0.947368  0.789474
4                        5  0.684211  0.631579
5                        6  0.789474  0.789474
6                        7  0.842105  0.842105
7                        8  0.789474  0.631579
8                        9  0.684211  0.526316
9                       10  0.684211  0.894737
10                      11  0.578947  0.631579
11                      12  0.578947  0.736842
Paired t-test: t = -0.625, p = 0.5446


In [14]:
import pandas as pd

df = pd.read_csv("scene_evaluation_responses_E3.csv")

# Binary yes
df["is_yes"] = (df["answer"].str.lower() == "yes").astype(int)

# --- participant alignment across systems (사람 매칭) ---
def map_participant_id(scene_id: int, system: str) -> int:
    # sys1-scene1 == sys2-scene7, sys1-scene7 == sys2-scene1 ...
    if system == "system1":
        return scene_id
    return scene_id + 6 if scene_id <= 6 else scene_id - 6

df["participant_id"] = df.apply(lambda r: map_participant_id(int(r["sceneId"]), r["systemType"]), axis=1)

# --- target_set은 sceneId로부터! (세트는 타겟 묶음) ---
df["target_set"] = df["sceneId"].apply(lambda x: 1 if int(x) <= 6 else 2)

# === 세트 내부에서 system1 vs system2 평균 비교 (질문별) ===
# 세트-시스템-질문 단위 yes 평균과 표본 수
agg = (df.groupby(["target_set","systemType","questionId"])["is_yes"]
         .agg(mean_yes="mean", n="count")
         .reset_index())

# 시스템별 열로 피벗
wide = (agg.pivot_table(index=["target_set","questionId"],
                        columns="systemType",
                        values=["mean_yes","n"])
           .reset_index())

# 컬럼 정리 (다중인덱스 -> 평평하게)
wide.columns = ["target_set","questionId",
                "mean_sys1","mean_sys2","n_sys1","n_sys2"]

# 차이 (System2 - System1)
wide["mean_diff"] = wide["mean_sys2"] - wide["mean_sys1"]

# 보기 좋은 정렬: 세트별로 절대차 큰 순
by_q_sorted = (wide.assign(abs_diff=lambda d: d["mean_diff"])
                    .sort_values(["target_set","abs_diff"], ascending=[True, True])
                    .drop(columns=["abs_diff"]))

print("=== (세트별) 질문별 System2 - System1 평균 yes비율 차이 ===")
print(by_q_sorted.to_string(index=False))

# 필요 시: 세트별로 System2가 더 못한 질문부터
by_q_worse2 = wide.sort_values(["target_set","mean_diff"], ascending=[True, True])
# print(by_q_worse2.to_string(index=False))


=== (세트별) 질문별 System2 - System1 평균 yes비율 차이 ===
 target_set  questionId  mean_sys1  mean_sys2  n_sys1  n_sys2  mean_diff
          1          10   0.833333   0.166667     6.0     6.0  -0.666667
          1           1   1.000000   0.500000     6.0     6.0  -0.500000
          1           6   0.333333   0.000000     6.0     6.0  -0.333333
          1           4   0.833333   0.666667     6.0     6.0  -0.166667
          1          11   0.666667   0.500000     6.0     6.0  -0.166667
          1           5   0.833333   0.833333     6.0     6.0   0.000000
          1           9   1.000000   1.000000     6.0     6.0   0.000000
          1          15   0.666667   0.666667     6.0     6.0   0.000000
          1          16   1.000000   1.000000     6.0     6.0   0.000000
          1          17   1.000000   1.000000     6.0     6.0   0.000000
          1          19   0.333333   0.333333     6.0     6.0   0.000000
          1           2   0.833333   1.000000     6.0     6.0   0.166667
   